# Multi-Table Metrics

This notebook measures **relational** quality on one Berka parent–child edge: **`account` → `loan`** (1-hop). It does not rerun single-table column metrics. Please refer to the evaluation directory for single-table quality notebook.

In [3]:
from pathlib import Path
import os


def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp')

In [ ]:
import json
import pickle
from logging import INFO

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

from midst_toolkit.common.logger import log
from midst_toolkit.evaluation.quality import CorrelationMatrixDifference
from implementations.tabular_data.evaluation.display_utils import log_metrics
from implementations.tabular_data.evaluation.preprocessing import syntheval_preprocess

ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"

PARENT_TABLE = "account"
CHILD_TABLE = "loan"
PK = "account_id"
FK = "account_id"

K_HOP = 1  # direct parent-child edge; this notebook does not walk longer paths


def drop_aux_columns(df: pd.DataFrame) -> pd.DataFrame:
    drop = [
        col
        for col in df.columns
        if col == "placeholder" or col.endswith("_cluster")
    ]
    return df.drop(columns=drop, errors="ignore")


def load_domain(table_name: str) -> dict:
    with open(base_data_dir / f"{table_name}_domain.json") as f:
        return json.load(f)


def feature_columns(domain: dict) -> tuple[list[str], list[str]]:
    """Split a domain file into numerical and categorical column names."""
    numerical = [name for name, spec in domain.items() if spec["type"] == "continuous"]
    categorical = [name for name, spec in domain.items() if spec["type"] == "discrete"]
    return numerical, categorical


real_parent = pd.read_csv(base_data_dir / f"{PARENT_TABLE}.csv")
real_child = pd.read_csv(base_data_dir / f"{CHILD_TABLE}.csv")

# For tables that don't require matching, the synthetic tables can be loaded from the before_matching folder.
pickle_path = base_output_dir / "before_matching" / "synthetic_tables.pkl"
with open(pickle_path, "rb") as f:
    synthetic_tables = pickle.load(f)

# ClavaDDPM writes keyed tables per edge; CSVs under exp_name drop *_id columns.
synthetic_parent = drop_aux_columns(synthetic_tables[("district", PARENT_TABLE)]["df"])
synthetic_child = drop_aux_columns(synthetic_tables[(PARENT_TABLE, CHILD_TABLE)]["df"])

parent_domain = load_domain(PARENT_TABLE)
child_domain = load_domain(CHILD_TABLE)

log(INFO, f"Example edge: {PARENT_TABLE} -> {CHILD_TABLE}")
log(INFO, f"Real parent {len(real_parent)} rows, child {len(real_child)} rows")
log(INFO, f"Synthetic parent {len(synthetic_parent)} rows, child {len(synthetic_child)} rows")
log(INFO, f"Loaded synthetic tables with keys from {pickle_path}")

INFO :      Example edge: account -> loan
INFO :      Real parent 4500 rows, child 682 rows
INFO :      Synthetic parent 4934 rows, child 725 rows
INFO :      Loaded synthetic tables with keys from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/before_matching/synthetic_tables.pkl


## 1) Within-table fidelity (per table)

Run the single-table evaluation metrics independently on each table. This is necessary but not sufficient as a multi-table synthesizer can generate high quality individual table while completely breaking the relationships between them.

Those metrics are in `implementations/tabular_data/evaluation/quality/quality_evaluation_pipeline.ipynb`. This notebook only scores **relationships** between `account` and `loan`.

## 2) Structural / referential integrity

- **Referential integrity**: % of foreign keys in child tables that resolve to a valid primary key in the parent table. This should be ~100% for a usable synthetic dataset.

- **Cardinality shape similarity**: for each parent-child pair, compare the distribution of "number of child rows per parent record" between real and synthetic

### Referential integrity

Fraction of `loan.account_id` values that exist in `account.account_id`. Closer to 1 is better.

In [6]:
def referential_integrity(
    parent: pd.DataFrame, child: pd.DataFrame, pk: str, fk: str
) -> dict[str, float]:
    valid = child[fk].isin(parent[pk])
    n_child = int(len(child))
    n_invalid = int((~valid).sum())
    return {
        "valid_rate": float(valid.mean()) if n_child else float("nan"),
        "n_invalid": n_invalid,
        "n_child": n_child,
        "n_parent": int(len(parent)),
    }


ri_real = referential_integrity(real_parent, real_child, PK, FK)
ri_synth = referential_integrity(synthetic_parent, synthetic_child, PK, FK)

ri_table = pd.DataFrame(
    [
        {"split": "real", **ri_real},
        {"split": "synthetic", **ri_synth},
    ]
)
log_metrics("REFERENTIAL INTEGRITY (account -> loan)", {"synthetic_valid_rate": ri_synth["valid_rate"]})
ri_table

INFO :      
REFERENTIAL INTEGRITY (account -> loan)
--------------------------------------------------------------------------------

INFO :      Metric: synthetic_valid_rate\tScore: 1.0


INFO :      --------------------------------------------------------------------------------



,split,valid_rate,n_invalid,n_child,n_parent
0,real,1.0,0,682,4500
1,synthetic,1.0,0,725,4934


### Cardinality shape similarity

For every parent row, count how many child rows point at it (including zeros). Compare those two count distributions with a two-sample Kolmogorov–Smirnov statistic.

- **`ks_distance`**: largest gap between the CDFs. Closer to 0 is better.
- **`ks_complement`**: `1 - ks_distance`. Closer to 1 is better (same orientation as SDV cardinality scores).

In [7]:
def children_per_parent(
    parent: pd.DataFrame, child: pd.DataFrame, pk: str, fk: str
) -> np.ndarray:
    counts = child.groupby(fk).size()
    return parent[pk].map(counts).fillna(0).to_numpy(dtype=float)


real_card = children_per_parent(real_parent, real_child, PK, FK)
synth_card = children_per_parent(synthetic_parent, synthetic_child, PK, FK)
ks_distance = float(ks_2samp(real_card, synth_card).statistic)

cardinality_results = {
    "ks_distance": ks_distance,
    "ks_complement": 1.0 - ks_distance,
    "real_mean_children": float(real_card.mean()),
    "synthetic_mean_children": float(synth_card.mean()),
    "real_frac_zero": float((real_card == 0).mean()),
    "synthetic_frac_zero": float((synth_card == 0).mean()),
}
log_metrics("CARDINALITY SHAPE (account -> loan)", cardinality_results)
pd.DataFrame([cardinality_results])

INFO :      
CARDINALITY SHAPE (account -> loan)
--------------------------------------------------------------------------------

INFO :      Metric: ks_distance\tScore: 0.004615952799171283
INFO :      Metric: ks_complement\tScore: 0.9953840472008287
INFO :      Metric: real_mean_children\tScore: 0.15155555555555555
INFO :      Metric: synthetic_mean_children\tScore: 0.14693960275638426
INFO :      Metric: real_frac_zero\tScore: 0.8484444444444444
INFO :      Metric: synthetic_frac_zero\tScore: 0.8530603972436157
INFO :      --------------------------------------------------------------------------------



,ks_distance,ks_complement,real_mean_children,synthetic_mean_children,real_frac_zero,synthetic_frac_zero
0,0.004616,0.995384,0.151556,0.14694,0.848444,0.85306


## 3) Cross-table (inter-table) trends

In the single-table evaluation, we looked at column pair trends. Specifically we looked at the statistical similarity between the real and synthetic data for pairs of columns (within the same table). This is often called pair-wise correlation or bivariate distributions of the columns. In multi-table settings, we can look at inter-table trends by comparing the column trends (pair-wise correlation) between different tables. For example a column between a parent table and a different column in a child table.

Source: https://docs.sdv.dev/sdv/evaluation/data-quality


The script uses a **k-hop join** with `k = 1`: inner-join `loan` to `account` on `account_id`, then scores **only parent–child column pairs** (not within-table pairs on the joined frame). The mixed-correlation machinery is the same as `CorrelationMatrixDifference` in the single-table pipeline (Pearson / Cramér’s V / correlation ratio). `corr_mat_diff` is the Frobenius norm of the difference of that rectangular block; closer to 0 is better.

In [8]:
from typing import Hashable


def prefix_features(df: pd.DataFrame, table_name: str, keep: list[str]) -> pd.DataFrame:
    rename = {col: f"{table_name}__{col}" for col in df.columns if col not in keep}
    return df.rename(columns=rename)


def hop_join(
    parent: pd.DataFrame,
    child: pd.DataFrame,
    pk: str,
    fk: str,
    parent_name: str,
    child_name: str,
) -> pd.DataFrame:
    """Inner-join child to parent (k=1). Join keys are dropped after the merge."""
    parent_pref = prefix_features(parent, parent_name, keep=[pk])
    child_pref = prefix_features(child, child_name, keep=[fk])
    # The inner merge is a standard child-grained join: each matching child row is kept, and the matching parent’s columns are copied onto that row.
    joined = child_pref.merge(parent_pref, left_on=fk, right_on=pk, how="inner")
    return joined.drop(columns=list[Hashable]({pk, fk}))


parent_num, parent_cat = feature_columns(parent_domain)
child_num, child_cat = feature_columns(child_domain)

real_joined = hop_join(real_parent, real_child, PK, FK, PARENT_TABLE, CHILD_TABLE)
synth_joined = hop_join(synthetic_parent, synthetic_child, PK, FK, PARENT_TABLE, CHILD_TABLE)

prefixed_parent_num = [f"{PARENT_TABLE}__{c}" for c in parent_num]
prefixed_parent_cat = [f"{PARENT_TABLE}__{c}" for c in parent_cat]
prefixed_child_num = [f"{CHILD_TABLE}__{c}" for c in child_num]
prefixed_child_cat = [f"{CHILD_TABLE}__{c}" for c in child_cat]

numerical_columns = prefixed_parent_num + prefixed_child_num
categorical_columns = prefixed_parent_cat + prefixed_child_cat
parent_feature_cols = prefixed_parent_num + prefixed_parent_cat
child_feature_cols = prefixed_child_num + prefixed_child_cat

feature_cols = numerical_columns + categorical_columns
real_joined = real_joined[feature_cols].copy()
synth_joined = synth_joined[feature_cols].copy()
for col in categorical_columns:
    real_joined[col] = real_joined[col].astype(str)
    synth_joined[col] = synth_joined[col].astype(str)

log(INFO, f"k-hop join (k={K_HOP}): real {real_joined.shape}, synthetic {synth_joined.shape}")

real_enc, synth_enc = syntheval_preprocess(
    numerical_columns, categorical_columns, real_joined, synth_joined
)

metric = CorrelationMatrixDifference(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    compute_mixed_correlations=True,
    do_preprocess=False,
)
# compute() scores the whole joined matrix, including within-table pairs. Re-evaluating with
# return_mats gives the matrices themselves, so we can keep only the parent x child block.
metric.compute(real_enc, synth_enc)
matrices = metric.syntheval_metric.evaluate(mixed_corr=True, return_mats=True)
real_block = matrices["real_cor_mat"].loc[parent_feature_cols, child_feature_cols]
synth_block = matrices["synt_cor_mat"].loc[parent_feature_cols, child_feature_cols]
block_diff = real_block - synth_block

cross_table_results = {
    "corr_mat_diff": float(np.linalg.norm(block_diff.to_numpy(dtype=float), ord="fro")),
    "n_column_pairs": float(block_diff.size),
    "n_parent_features": float(len(parent_feature_cols)),
    "n_child_features": float(len(child_feature_cols)),
}
log_metrics("K-HOP PAIRWISE CORRELATION (account columns x loan columns)", cross_table_results)
print("Parent x child correlation difference (real − synthetic):")
block_diff

INFO :      k-hop join (k=1): real (682, 7), synthetic (725, 7)


INFO :      
K-HOP PAIRWISE CORRELATION (account columns x loan columns)
--------------------------------------------------------------------------------

INFO :      Metric: corr_mat_diff\tScore: 0.9658033981185801
INFO :      Metric: n_column_pairs\tScore: 10.0
INFO :      Metric: n_parent_features\tScore: 2.0
INFO :      Metric: n_child_features\tScore: 5.0
INFO :      --------------------------------------------------------------------------------



Parent x child correlation difference (real − synthetic):


,loan__loan_date,loan__amount,loan__duration,loan__payments,loan__status
account__account_date,0.899748,-0.012905,0.070231,-0.080870,0.334015
account__frequency,0.002950,0.000761,0.000737,0.003799,0.000143
